# Qdislib — 03 · Backends and interop

Qdislib is **backend-agnostic**: the same workflow drives Qibo, Qiskit and
CUDA-Q — only the `software=` argument changes. Any framework that exports
OpenQASM 2.0 can also feed in via `from_qasm`.

In [ ]:
from qibo import models, gates


def example_circuit():
    """A small 5-qubit circuit: cuttable (CZ chain), non-trivial <ZZZZZ> ~= 0.44."""
    circuit = models.Circuit(5)
    circuit.add(gates.RY(0, 0.8))
    circuit.add(gates.CZ(0, 1))
    circuit.add(gates.RY(1, 0.5))
    circuit.add(gates.CZ(1, 2))
    circuit.add(gates.RY(2, 0.6))
    circuit.add(gates.CZ(2, 3))
    circuit.add(gates.RY(3, 0.4))
    circuit.add(gates.CZ(3, 4))
    circuit.add(gates.RX(4, 0.3))
    return circuit


circuit = example_circuit()
print(circuit.draw())

In [ ]:
import Qdislib as qd

circuit = example_circuit()
cut = qd.find_cut(circuit, gate_cut=True, wire_cut=False)
print("cut:", cut)

## Qibo

In [ ]:
print("qibo  :", qd.gate_cutting(example_circuit(), cut, software="qibo", shots=8192))

## Qiskit

Same circuit and cut, executed on Qiskit Aer (`pip install Qdislib[qiskit]`).

In [ ]:
print("qiskit:", qd.gate_cutting(example_circuit(), cut, software="qiskit", shots=8192))

## CUDA-Q

With `pip install Qdislib[cudaq]` you can build the circuit as a CUDA-Q
kernel and run with `software="cudaq"` (GPU when available, CPU otherwise).
`find_cut` works directly on a kernel too.

In [ ]:
import cudaq
import Qdislib as qd

@cudaq.kernel
def kernel():
    q = cudaq.qvector(5)
    ry(0.8, q[0])
    z.ctrl(q[0], q[1])
    ry(0.5, q[1])
    z.ctrl(q[1], q[2])
    ry(0.6, q[2])
    z.ctrl(q[2], q[3])
    ry(0.4, q[3])
    z.ctrl(q[3], q[4])
    rx(0.3, q[4])

cut = qd.find_cut(kernel, gate_cut=True, wire_cut=False)
print(qd.gate_cutting(kernel, cut, software="cudaq", shots=8192))

## Interop: `from_qasm`

Bring circuits from Cirq, PennyLane, Braket, ... through their OpenQASM 2.0
export. Here we round-trip a Qibo circuit through QASM into a Qiskit one
and cut that.

In [ ]:
qasm = example_circuit().to_qasm()
qc = qd.from_qasm(qasm, software="qiskit")
cut = qd.find_cut(qc, gate_cut=True, wire_cut=False)
print("cut on the imported circuit:", cut)
print("value:", qd.gate_cutting(qd.from_qasm(qasm, software="qiskit"), cut, software="qiskit", shots=8192))

---
Next: **04 · Observables** — beyond `<ZZZZZ>`.